In [ ]:
# Configuration
import os
from pyspark.sql import SparkSession, functions as F, Window

IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IS_DATABRICKS:
    dbutils.widgets.text('cur_bucket', 'flex-cur-data')
    dbutils.widgets.text('year', '2026')
    dbutils.widgets.text('month', '05')
    CUR_BUCKET = dbutils.widgets.get('cur_bucket')
    YEAR = dbutils.widgets.get('year')
    MONTH = dbutils.widgets.get('month')
    STAGING_PATH = f's3://{CUR_BUCKET}/staging/{YEAR}/{MONTH}/'
else:
    CUR_BUCKET = os.environ.get('CUR_BUCKET', 'flex-cur-data')
    S3_ENDPOINT = os.environ.get('AWS_ENDPOINT_URL', 'http://localhost:4566')
    YEAR = '2026'
    MONTH = '05'
    STAGING_PATH = f's3a://{CUR_BUCKET}/staging/{YEAR}/{MONTH}/'

if not IS_DATABRICKS:
    spark = (SparkSession.builder
        .master('local[*]')
        .appName('Flex-CUR-Transform')
        .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262')
        .config('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT)
        .config('spark.hadoop.fs.s3a.access.key', 'test')
        .config('spark.hadoop.fs.s3a.secret.key', 'test')
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.driver.memory', '4g')
        .getOrCreate())
else:
    spark = SparkSession.builder.getOrCreate()

print(f'Spark ready: {spark.version}')

In [ ]:
# Read Staged Data
try:
    staged_df = spark.read.parquet(STAGING_PATH)
except:
    local_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'staging', f'{YEAR}_{MONTH}')
    staged_df = spark.read.parquet(local_path)

print(f'Staged records: {staged_df.count():,}')

# Alias columns for easier use
df = (staged_df
    .withColumnRenamed('custom/BusinessUnitId', 'business_unit_id')
    .withColumnRenamed('custom/BusinessUnitName', 'business_unit_name')
    .withColumnRenamed('custom/CostCenter', 'cost_center')
    .withColumnRenamed('lineItem/ProductCode', 'service')
    .withColumnRenamed('product/region', 'region')
    .withColumn('date', F.to_date('usage_date'))
)

In [ ]:
# ===========================
# TRANSFORM 1: KPIs per Business Unit
# ===========================

# Current month total spend per BU
kpi_spend = (df
    .groupBy('business_unit_id')
    .agg(
        F.sum('cost').alias('total_spend'),
        F.countDistinct('lineItem/ResourceId').alias('active_resources'),
        F.avg('usage_amount').alias('avg_usage'),
    )
)

# Compute spend change % (compare first half vs second half of month as proxy)
mid_month = f'{YEAR}-{MONTH}-15'
first_half = df.filter(F.col('date') < mid_month).groupBy('business_unit_id').agg(F.sum('cost').alias('first_half_spend'))
second_half = df.filter(F.col('date') >= mid_month).groupBy('business_unit_id').agg(F.sum('cost').alias('second_half_spend'))

kpi_df = (kpi_spend
    .join(first_half, 'business_unit_id', 'left')
    .join(second_half, 'business_unit_id', 'left')
    .withColumn('spend_change_pct', 
        F.round(((F.col('second_half_spend') - F.col('first_half_spend')) / F.col('first_half_spend')) * 100, 2)
    )
    .withColumn('period', F.lit(f'{YEAR}-{MONTH}-01').cast('date'))
    .select('business_unit_id', 'period', 'total_spend', 'spend_change_pct', 'active_resources')
)

print('KPIs per BU:')
kpi_df.show(truncate=False)

In [ ]:
# ===========================
# TRANSFORM 2: Cloud Usage Time-Series (daily by service category)
# ===========================

# Map services to categories
SERVICE_CATEGORIES = {
    'AmazonEC2': 'compute', 'AmazonEKS': 'compute', 'AWSLambda': 'compute',
    'AmazonS3': 'storage',
    'AWSDataTransfer': 'network',
    'AmazonRDS': 'database', 'AmazonRedshift': 'database', 'AmazonElastiCache': 'database',
}

# Create mapping expression
category_expr = F.coalesce(
    *[F.when(F.col('service') == k, F.lit(v)) for k, v in SERVICE_CATEGORIES.items()],
    F.lit('other')
)

usage_ts_df = (df
    .withColumn('category', category_expr)
    .groupBy('business_unit_id', 'date', 'category')
    .agg(F.sum('cost').alias('daily_cost'))
    .orderBy('business_unit_id', 'date', 'category')
)

# Pivot to get compute/storage/network/database as columns (matches CloudUsagePoint type)
usage_pivoted_df = (usage_ts_df
    .groupBy('business_unit_id', 'date')
    .pivot('category', ['compute', 'storage', 'network', 'database', 'other'])
    .agg(F.sum('daily_cost'))
    .fillna(0)
    .withColumnRenamed('date', 'time')
)

print('Cloud Usage Time-Series (sample):')
usage_pivoted_df.show(5, truncate=False)

In [ ]:
# ===========================
# TRANSFORM 3: Chargeback per Team/Cost Center
# ===========================

# Extract team from tags
chargeback_df = (df
    .withColumn('team', F.coalesce(
        F.get_json_object(F.to_json('resourceTags'), '$."user:Team"'),
        F.lit('Untagged')
    ))
    .withColumn('owner', F.coalesce(
        F.get_json_object(F.to_json('resourceTags'), '$."user:Owner"'),
        F.lit('unknown')
    ))
    .groupBy('business_unit_id', 'cost_center', 'team', 'owner')
    .agg(
        F.sum('cost').alias('monthly_spend'),
        F.countDistinct('lineItem/ResourceId').alias('resource_count'),
    )
    .withColumn('period', F.lit(f'{YEAR}-{MONTH}-01').cast('date'))
    .withColumn('cost_per_engineer', F.lit(0.0))  # Will be enriched with workforce data
)

print('Chargeback summary:')
chargeback_df.show(5, truncate=False)

In [ ]:
# ===========================
# TRANSFORM 4: Tag Compliance Scoring
# ===========================

REQUIRED_TAGS = ['user:Team', 'user:CostCenter', 'user:Environment', 'user:Owner']

# Count how many required tags are present per resource
tag_compliance_df = df.select(
    'business_unit_id',
    'lineItem/ResourceId',
    'resourceTags'
).dropDuplicates(['lineItem/ResourceId'])

# Calculate compliance per resource
tags_json = F.to_json('resourceTags')
for tag in REQUIRED_TAGS:
    tag_compliance_df = tag_compliance_df.withColumn(
        f'has_{tag.replace(":", "_")}',
        F.when(F.get_json_object(tags_json, f'$."{tag}"').isNotNull(), 1).otherwise(0)
    )

tag_cols = [f'has_{t.replace(":", "_")}' for t in REQUIRED_TAGS]
tag_compliance_df = tag_compliance_df.withColumn(
    'compliance_score',
    sum(F.col(c) for c in tag_cols) / len(REQUIRED_TAGS) * 100
)

# Aggregate per BU
bu_tag_compliance = (tag_compliance_df
    .groupBy('business_unit_id')
    .agg(
        F.avg('compliance_score').alias('avg_tag_compliance_pct'),
        F.count('*').alias('total_resources'),
        F.sum(F.when(F.col('compliance_score') == 100, 1).otherwise(0)).alias('fully_compliant')
    )
)

print('Tag Compliance by BU:')
bu_tag_compliance.show(truncate=False)

In [ ]:
# ===========================
# TRANSFORM 5: Savings Opportunities Detection
# ===========================

# Detect underutilized resources (usage < 30% of allocated)
# and idle resources (zero usage for 7+ days)

resource_usage = (df
    .groupBy('business_unit_id', 'lineItem/ResourceId', 'service')
    .agg(
        F.sum('cost').alias('total_cost'),
        F.avg('usage_amount').alias('avg_usage'),
        F.count('*').alias('days_active'),
        F.max('date').alias('last_active'),
    )
)

# Days in the month
from datetime import datetime
import calendar
days_in_month = calendar.monthrange(int(YEAR), int(MONTH))[1]

# Idle resources: active less than 60% of the month
idle_resources = (resource_usage
    .filter(F.col('days_active') < days_in_month * 0.6)
    .filter(F.col('total_cost') > 50)  # Only flag if meaningful cost
    .withColumn('category', F.lit('idle_resources'))
    .withColumn('monthly_savings', F.col('total_cost') * 0.7)  # Estimate 70% saveable
    .withColumn('confidence', F.lit(0.8))
    .withColumn('effort', F.lit('low'))
    .select('business_unit_id', 'lineItem/ResourceId', 'service', 'category', 
            'monthly_savings', 'confidence', 'effort', 'total_cost')
)

print(f'Savings opportunities detected: {idle_resources.count()}')
idle_resources.show(5, truncate=False)

In [ ]:
# ===========================
# Store transformed DataFrames as temp views for Notebook 3
# ===========================

kpi_df.createOrReplaceTempView('transformed_kpis')
usage_pivoted_df.createOrReplaceTempView('transformed_cloud_usage')
chargeback_df.createOrReplaceTempView('transformed_chargeback')
bu_tag_compliance.createOrReplaceTempView('transformed_tag_compliance')
idle_resources.createOrReplaceTempView('transformed_savings')

# Also save to local parquet for Notebook 3 if running standalone
output_base = os.path.join(os.path.dirname(os.getcwd()), 'data', 'transformed', f'{YEAR}_{MONTH}')
os.makedirs(output_base, exist_ok=True)

kpi_df.toPandas().to_parquet(f'{output_base}/kpis.parquet')
usage_pivoted_df.toPandas().to_parquet(f'{output_base}/cloud_usage.parquet')
chargeback_df.toPandas().to_parquet(f'{output_base}/chargeback.parquet')
bu_tag_compliance.toPandas().to_parquet(f'{output_base}/tag_compliance.parquet')
idle_resources.toPandas().to_parquet(f'{output_base}/savings.parquet')

print(f'✅ All transforms saved to: {output_base}')
print(f'   - kpis.parquet')
print(f'   - cloud_usage.parquet')
print(f'   - chargeback.parquet')
print(f'   - tag_compliance.parquet')
print(f'   - savings.parquet')